<a href="https://colab.research.google.com/github/mobarakol/tutorial_notebooks/blob/main/GPT2_CN_Hancock_25F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import os
import random
import numpy as np
import pandas as pd
import json

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

from transformers import (
    GPT2Tokenizer,
    GPT2Model
)

# ------------------------------------------------------------
# Read Data
# ------------------------------------------------------------

import gdown
import json
import pandas as pd

# Clinical data
gdown.download(
    id="1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q",
    output="clinical_data.json",
    quiet=False
)

# Pathological data
gdown.download(
    id="1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F",
    output="pathological_data.json",
    quiet=False
)

with open("clinical_data.json", "r") as f:
    clinical = pd.DataFrame(json.load(f))

with open("pathological_data.json", "r") as f:
    pathology = pd.DataFrame(json.load(f))

    df = clinical.merge(
    pathology,
    on='patient_id',
    how='inner'
)

print(df.shape)

print("Clinical shape:", clinical.shape)
print("Pathology shape:", pathology.shape)

# ============================================================
# HPV Labels
# ============================================================

df = df[
    df['hpv_association_p16'].isin(
        ['positive', 'negative']
    )
].copy()

df['HPV'] = df['hpv_association_p16'].map({
    'negative': 0,
    'positive': 1
})

# ============================================================
# GPT2 Hancock Features
# ============================================================

features = [
    'age_at_initial_diagnosis',
    'sex',
    'smoking_status',
    'primarily_metastasis',
    'first_treatment_intent',
    'first_treatment_modality',
    'days_to_first_treatment',
    'adjuvant_treatment_intent',
    'adjuvant_radiotherapy',
    'adjuvant_radiotherapy_modality',
    'adjuvant_systemic_therapy',
    'adjuvant_systemic_therapy_modality',
    'adjuvant_radiochemotherapy',
    'primary_tumor_site',
    'pT_stage',
    'pN_stage',
    'histologic_type',
    'number_of_positive_lymph_nodes',
    'number_of_resected_lymph_nodes',
    'perinodal_invasion',
    'lymphovascular_invasion_L',
    'vascular_invasion_V',
    'perineural_invasion_Pn',
    'resection_status',
    'infiltration_depth_in_mm'
]

# ============================================================
# Missing Values
# ============================================================

for col in features:

    if df[col].dtype == object:

        df[col] = df[col].fillna(
            df[col].mode()[0]
        )

    else:

        df[col] = df[col].fillna(
            df[col].median()
        )

# ============================================================
# Narrative Function
# ============================================================

def patient_to_text(row):

    text = (
        f"This patient was diagnosed at age {row['age_at_initial_diagnosis']}. "
        f"Sex is {row['sex']}. "
        f"Smoking status is {row['smoking_status']}. "
        f"Primary tumour site is {row['primary_tumor_site']}. "
        f"Histologic type is {row['histologic_type']}. "
        f"Pathological T stage is {row['pT_stage']}. "
        f"Pathological N stage is {row['pN_stage']}. "
        f"The patient has "
        f"{row['number_of_positive_lymph_nodes']} positive lymph nodes and "
        f"{row['number_of_resected_lymph_nodes']} resected lymph nodes. "
        f"Perinodal invasion status is {row['perinodal_invasion']}. "
        f"Lymphovascular invasion status is {row['lymphovascular_invasion_L']}. "
        f"Vascular invasion status is {row['vascular_invasion_V']}. "
        f"Perineural invasion status is {row['perineural_invasion_Pn']}. "
        f"Resection status is {row['resection_status']}. "
        f"Infiltration depth is {row['infiltration_depth_in_mm']} mm. "
        f"First treatment intent was {row['first_treatment_intent']}. "
        f"First treatment modality was {row['first_treatment_modality']}. "
        f"Days to first treatment was {row['days_to_first_treatment']}. "
        f"Adjuvant treatment intent was {row['adjuvant_treatment_intent']}. "
        f"Adjuvant radiotherapy was {row['adjuvant_radiotherapy']}. "
        f"Adjuvant radiotherapy modality was {row['adjuvant_radiotherapy_modality']}. "
        f"Adjuvant systemic therapy was {row['adjuvant_systemic_therapy']}. "
        f"Adjuvant systemic therapy modality was {row['adjuvant_systemic_therapy_modality']}. "
        f"Adjuvant radiochemotherapy was {row['adjuvant_radiochemotherapy']}. "
        f"Predict whether the patient is HPV positive or HPV negative."
    )

    return text

# ============================================================
# Create GPT2 Text Corpus
# ============================================================

texts = df.apply(
    patient_to_text,
    axis=1
).tolist()

labels = df['HPV'].tolist()

print("Patients:", len(texts))
print(pd.Series(labels).value_counts())






# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


SEED = 42
seed_everything(SEED)


# ============================================================
# Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)




# ============================================================
# Train-test split
# ============================================================

train_texts, test_texts, y_train, y_test = (
    train_test_split(
        texts,
        labels,
        test_size=0.2,
        random_state=SEED,
        stratify=labels
    )
)

print("\nTrain class distribution:")
print(pd.Series(y_train).value_counts())

print("\nTest class distribution:")
print(pd.Series(y_test).value_counts())


# ============================================================
# Tokenizer
# ============================================================

MODEL_NAME = "openai-community/gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(
    MODEL_NAME
)

# GPT-2 has no default padding token
tokenizer.pad_token = tokenizer.eos_token

MAX_LENGTH = 128


# ============================================================
# Dataset
# ============================================================

class HPVTextDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length=128
    ):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):

        encoding = self.tokenizer(
            self.texts[index],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": (
                encoding["input_ids"].squeeze(0)
            ),
            "attention_mask": (
                encoding["attention_mask"].squeeze(0)
            ),
            "label": torch.tensor(
                self.labels[index],
                dtype=torch.long
            )
        }


train_dataset = HPVTextDataset(
    texts=train_texts,
    labels=y_train,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

test_dataset = HPVTextDataset(
    texts=test_texts,
    labels=y_test,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)


# ============================================================
# DataLoaders
# ============================================================

BATCH_SIZE = 256

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


# ============================================================
# GPT-2 text classifier
# ============================================================

class HPVNetGPT2Text(nn.Module):

    def __init__(
        self,
        model_name="openai-community/gpt2",
        num_classes=2,
        dropout=0.1,
        freeze_gpt2=True
    ):
        super().__init__()

        self.freeze_gpt2 = freeze_gpt2

        self.gpt2 = GPT2Model.from_pretrained(
            model_name
        )

        self.gpt2.config.use_cache = False
        self.gpt2.config.pad_token_id = (
            tokenizer.pad_token_id
        )

        hidden_size = self.gpt2.config.hidden_size

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

        if freeze_gpt2:
            self.gpt2.requires_grad_(False)

    def train(self, mode=True):
        super().train(mode)

        if self.freeze_gpt2:
            self.gpt2.eval()

        return self

    def forward(
        self,
        input_ids,
        attention_mask
    ):
        outputs = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True
        )

        hidden_states = outputs.last_hidden_state

        # Masked mean pooling
        expanded_mask = (
            attention_mask
            .unsqueeze(-1)
            .expand_as(hidden_states)
            .float()
        )

        hidden_sum = (
            hidden_states * expanded_mask
        ).sum(dim=1)

        valid_token_count = (
            expanded_mask.sum(dim=1)
            .clamp(min=1e-9)
        )

        pooled_output = (
            hidden_sum / valid_token_count
        )

        logits = self.classifier(
            pooled_output
        )

        return logits


# ============================================================
# Model
# ============================================================

model = HPVNetGPT2Text(
    model_name=MODEL_NAME,
    num_classes=2,
    freeze_gpt2=True
).to(device)


# ============================================================
# Class weights
# ============================================================

class_counts = np.bincount(y_train)

class_weights = (
    len(y_train)
    / (
        len(class_counts) *
        class_counts
    )
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

print("\nClass weights:", class_weights)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


# ============================================================
# Optimiser
# ============================================================

optimizer = torch.optim.AdamW(
    filter(
        lambda parameter: parameter.requires_grad,
        model.parameters()
    ),
    lr=1e-3,
    weight_decay=1e-4
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
)


# ============================================================
# Training
# ============================================================

EPOCHS = 100
best_val_loss = float("inf")
best_epoch = 0

for epoch in range(EPOCHS):

    model.train()

    train_loss_sum = 0.0
    train_sample_count = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        train_loss = criterion(
            outputs,
            labels
        )

        train_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            filter(
                lambda parameter: parameter.requires_grad,
                model.parameters()
            ),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum += (
            train_loss.item() *
            input_ids.size(0)
        )

        train_sample_count += (
            input_ids.size(0)
        )

    average_train_loss = (
        train_loss_sum /
        train_sample_count
    )

    model.eval()

    test_loss_sum = 0.0
    test_sample_count = 0

    with torch.no_grad():

        for batch in test_loader:

            input_ids = batch[
                "input_ids"
            ].to(
                device,
                non_blocking=True
            )

            attention_mask = batch[
                "attention_mask"
            ].to(
                device,
                non_blocking=True
            )

            labels = batch["label"].to(
                device,
                non_blocking=True
            )

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            test_loss = criterion(
                outputs,
                labels
            )

            test_loss_sum += (
                test_loss.item() *
                input_ids.size(0)
            )

            test_sample_count += (
                input_ids.size(0)
            )

    average_test_loss = (
        test_loss_sum /
        test_sample_count
    )

    if average_test_loss < best_val_loss:

        best_val_loss = average_test_loss
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "best_model_gpt2_text.pth"
        )

    if (epoch + 1) % 5 == 0:

        print(
            f"Epoch {epoch + 1:03d}, "
            f"Train={average_train_loss:.4f}, "
            f"Test={average_test_loss:.4f}, "
            f"Best Epoch={best_epoch}"
        )


print("\nBest Test Loss:", best_val_loss)
print("Best Epoch:", best_epoch)


# ============================================================
# Load best model
# ============================================================

checkpoint = torch.load(
    "best_model_gpt2_text.pth",
    map_location=device
)

model.load_state_dict(checkpoint)
model.eval()


# ============================================================
# Inference
# ============================================================

all_probabilities = []
all_predictions = []
all_targets = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )

        labels = batch["label"]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        all_probabilities.append(
            probabilities.cpu()
        )

        all_predictions.append(
            predictions.cpu()
        )

        all_targets.append(
            labels.cpu()

                    )


probabilities = torch.cat(
    all_probabilities,
    dim=0
).numpy()

predicted = torch.cat(
    all_predictions,
    dim=0
).numpy()

y_true = torch.cat(
    all_targets,
    dim=0
).numpy()


# ============================================================
# Evaluation
# ============================================================

results = classification_report(
    y_true,
    predicted,
    digits=4,
    zero_division=0
)

print("\nClassification report:")
print(results)

bal_acc = balanced_accuracy_score(
    y_true,
    predicted
)

f1 = f1_score(
    y_true,
    predicted,
    zero_division=0
)

auc = roc_auc_score(
    y_true,
    probabilities[:, 1]
)

print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"AUC:               {auc:.4f}")

Downloading...
From: https://drive.google.com/uc?id=1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q
To: /content/clinical_data.json
100%|██████████| 810k/810k [00:00<00:00, 123MB/s]
Downloading...
From: https://drive.google.com/uc?id=1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F
To: /content/pathological_data.json
100%|██████████| 458k/458k [00:00<00:00, 88.9MB/s]


(763, 49)
Clinical shape: (763, 32)
Pathology shape: (763, 18)
Patients: 332
0    191
1    141
Name: count, dtype: int64
Using device: cuda

Train class distribution:
0    152
1    113
Name: count, dtype: int64

Test class distribution:
0    39
1    28
Name: count, dtype: int64


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Class weights: tensor([0.8717, 1.1726], device='cuda:0')
Trainable parameters: 100226
Epoch 005, Train=0.7408, Test=0.7079, Best Epoch=5
Epoch 010, Train=0.6945, Test=0.6943, Best Epoch=9
Epoch 015, Train=0.6925, Test=0.6945, Best Epoch=9
Epoch 020, Train=0.6952, Test=0.6906, Best Epoch=19
Epoch 025, Train=0.6874, Test=0.6910, Best Epoch=22
Epoch 030, Train=0.6937, Test=0.6873, Best Epoch=30
Epoch 035, Train=0.6910, Test=0.6916, Best Epoch=30
Epoch 040, Train=0.6916, Test=0.6856, Best Epoch=40
Epoch 045, Train=0.6850, Test=0.6923, Best Epoch=43
Epoch 050, Train=0.6833, Test=0.6884, Best Epoch=48
Epoch 055, Train=0.6877, Test=0.6824, Best Epoch=55
Epoch 060, Train=0.6853, Test=0.6820, Best Epoch=60
Epoch 065, Train=0.6864, Test=0.6832, Best Epoch=60
Epoch 070, Train=0.6888, Test=0.6806, Best Epoch=66
Epoch 075, Train=0.6848, Test=0.6782, Best Epoch=75
Epoch 080, Train=0.6820, Test=0.6768, Best Epoch=78
Epoch 085, Train=0.6766, Test=0.6729, Best Epoch=85
Epoch 090, Train=0.6816, Test=0.

In [ ]:

import os
import random
import numpy as np
import pandas as pd
import json

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

from transformers import (
    GPT2Tokenizer,
    GPT2Model
)

# ------------------------------------------------------------
# Read Data
# ------------------------------------------------------------

import gdown
import json
import pandas as pd

# Clinical data
gdown.download(
    id="1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q",
    output="clinical_data.json",
    quiet=False
)

# Pathological data
gdown.download(
    id="1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F",
    output="pathological_data.json",
    quiet=False
)

with open("clinical_data.json", "r") as f:
    clinical = pd.DataFrame(json.load(f))

with open("pathological_data.json", "r") as f:
    pathology = pd.DataFrame(json.load(f))

    df = clinical.merge(
    pathology,
    on='patient_id',
    how='inner'
)

print(df.shape)

print("Clinical shape:", clinical.shape)
print("Pathology shape:", pathology.shape)

# ============================================================
# HPV Labels
# ============================================================

df = df[
    df['hpv_association_p16'].isin(
        ['positive', 'negative']
    )
].copy()

df['HPV'] = df['hpv_association_p16'].map({
    'negative': 0,
    'positive': 1
})

# ============================================================
# GPT2 Hancock Features
# ============================================================

features = [
    'age_at_initial_diagnosis',
    'sex',
    'smoking_status',
    'primarily_metastasis',
    'first_treatment_intent',
    'first_treatment_modality',
    'days_to_first_treatment',
    'adjuvant_treatment_intent',
    'adjuvant_radiotherapy',
    'adjuvant_radiotherapy_modality',
    'adjuvant_systemic_therapy',
    'adjuvant_systemic_therapy_modality',
    'adjuvant_radiochemotherapy',
    'primary_tumor_site',
    'pT_stage',
    'pN_stage',
    'histologic_type',
    'number_of_positive_lymph_nodes',
    'number_of_resected_lymph_nodes',
    'perinodal_invasion',
    'lymphovascular_invasion_L',
    'vascular_invasion_V',
    'perineural_invasion_Pn',
    'resection_status',
    'infiltration_depth_in_mm'
]

# ============================================================
# Missing Values
# ============================================================

for col in features:

    if df[col].dtype == object:

        df[col] = df[col].fillna(
            df[col].mode()[0]
        )

    else:

        df[col] = df[col].fillna(
            df[col].median()
        )

# ============================================================
# Narrative Function
# ============================================================

def patient_to_text(row):

    text = (
        f"This patient was diagnosed at age {row['age_at_initial_diagnosis']}. "
        f"Sex is {row['sex']}. "
        f"Smoking status is {row['smoking_status']}. "
        f"Primary tumour site is {row['primary_tumor_site']}. "
        f"Histologic type is {row['histologic_type']}. "
        f"Pathological T stage is {row['pT_stage']}. "
        f"Pathological N stage is {row['pN_stage']}. "
        f"The patient has "
        f"{row['number_of_positive_lymph_nodes']} positive lymph nodes and "
        f"{row['number_of_resected_lymph_nodes']} resected lymph nodes. "
        f"Perinodal invasion status is {row['perinodal_invasion']}. "
        f"Lymphovascular invasion status is {row['lymphovascular_invasion_L']}. "
        f"Vascular invasion status is {row['vascular_invasion_V']}. "
        f"Perineural invasion status is {row['perineural_invasion_Pn']}. "
        f"Resection status is {row['resection_status']}. "
        f"Infiltration depth is {row['infiltration_depth_in_mm']} mm. "
        f"First treatment intent was {row['first_treatment_intent']}. "
        f"First treatment modality was {row['first_treatment_modality']}. "
        f"Days to first treatment was {row['days_to_first_treatment']}. "
        f"Adjuvant treatment intent was {row['adjuvant_treatment_intent']}. "
        f"Adjuvant radiotherapy was {row['adjuvant_radiotherapy']}. "
        f"Adjuvant radiotherapy modality was {row['adjuvant_radiotherapy_modality']}. "
        f"Adjuvant systemic therapy was {row['adjuvant_systemic_therapy']}. "
        f"Adjuvant systemic therapy modality was {row['adjuvant_systemic_therapy_modality']}. "
        f"Adjuvant radiochemotherapy was {row['adjuvant_radiochemotherapy']}. "
        f"Predict whether the patient is HPV positive or HPV negative."
    )

    return text

# ============================================================
# Create GPT2 Text Corpus
# ============================================================

texts = df.apply(
    patient_to_text,
    axis=1
).tolist()

labels = df['HPV'].tolist()

print("Patients:", len(texts))
print(pd.Series(labels).value_counts())






# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


SEED = 42
seed_everything(SEED)


# ============================================================
# Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)




# ============================================================
# Train-test split
# ============================================================

train_texts, test_texts, y_train, y_test = (
    train_test_split(
        texts,
        labels,
        test_size=0.2,
        random_state=SEED,
        stratify=labels
    )
)

print("\nTrain class distribution:")
print(pd.Series(y_train).value_counts())

print("\nTest class distribution:")
print(pd.Series(y_test).value_counts())


# ============================================================
# Tokenizer
# ============================================================

MODEL_NAME = "openai-community/gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(
    MODEL_NAME
)

# GPT-2 has no default padding token
tokenizer.pad_token = tokenizer.eos_token

MAX_LENGTH = 256


# ============================================================
# Dataset
# ============================================================

class HPVTextDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length=128
    ):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):

        encoding = self.tokenizer(
            self.texts[index],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": (
                encoding["input_ids"].squeeze(0)
            ),
            "attention_mask": (
                encoding["attention_mask"].squeeze(0)
            ),
            "label": torch.tensor(
                self.labels[index],
                dtype=torch.long
            )
        }


train_dataset = HPVTextDataset(
    texts=train_texts,
    labels=y_train,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

test_dataset = HPVTextDataset(
    texts=test_texts,
    labels=y_test,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)


# ============================================================
# DataLoaders
# ============================================================

BATCH_SIZE = 256

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


# ============================================================
# GPT-2 text classifier
# ============================================================

class HPVNetGPT2Text(nn.Module):

    def __init__(
        self,
        model_name="openai-community/gpt2",
        num_classes=2,
        dropout=0.1,
        freeze_gpt2=True
    ):
        super().__init__()

        self.freeze_gpt2 = freeze_gpt2

        self.gpt2 = GPT2Model.from_pretrained(
            model_name
        )

        self.gpt2.config.use_cache = False
        self.gpt2.config.pad_token_id = (
            tokenizer.pad_token_id
        )

        hidden_size = self.gpt2.config.hidden_size

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

        if freeze_gpt2:
            self.gpt2.requires_grad_(False)

    def train(self, mode=True):
        super().train(mode)

        if self.freeze_gpt2:
            self.gpt2.eval()

        return self

    def forward(
        self,
        input_ids,
        attention_mask
    ):
        outputs = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True
        )

        hidden_states = outputs.last_hidden_state

        # Masked mean pooling
        expanded_mask = (
            attention_mask
            .unsqueeze(-1)
            .expand_as(hidden_states)
            .float()
        )

        hidden_sum = (
            hidden_states * expanded_mask
        ).sum(dim=1)

        valid_token_count = (
            expanded_mask.sum(dim=1)
            .clamp(min=1e-9)
        )

        pooled_output = (
            hidden_sum / valid_token_count
        )

        logits = self.classifier(
            pooled_output
        )

        return logits


# ============================================================
# Model
# ============================================================

model = HPVNetGPT2Text(
    model_name=MODEL_NAME,
    num_classes=2,
    freeze_gpt2=True
).to(device)


# ============================================================
# Class weights
# ============================================================

class_counts = np.bincount(y_train)

class_weights = (
    len(y_train)
    / (
        len(class_counts) *
        class_counts
    )
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

print("\nClass weights:", class_weights)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# criterion = nn.CrossEntropyLoss()


# ============================================================
# Optimiser
# ============================================================

optimizer = torch.optim.AdamW(
    filter(
        lambda parameter: parameter.requires_grad,
        model.parameters()
    ),
    lr=1e-3,
    weight_decay=1e-4
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
)


# ============================================================
# Training
# ============================================================

EPOCHS = 200
best_val_loss = float("inf")
best_epoch = 0

for epoch in range(EPOCHS):

    model.train()

    train_loss_sum = 0.0
    train_sample_count = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        train_loss = criterion(
            outputs,
            labels
        )

        train_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            filter(
                lambda parameter: parameter.requires_grad,
                model.parameters()
            ),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum += (
            train_loss.item() *
            input_ids.size(0)
        )

        train_sample_count += (
            input_ids.size(0)
        )

    average_train_loss = (
        train_loss_sum /
        train_sample_count
    )

    model.eval()

    test_loss_sum = 0.0
    test_sample_count = 0

    with torch.no_grad():

        for batch in test_loader:

            input_ids = batch[
                "input_ids"
            ].to(
                device,
                non_blocking=True
            )

            attention_mask = batch[
                "attention_mask"
            ].to(
                device,
                non_blocking=True
            )

            labels = batch["label"].to(
                device,
                non_blocking=True
            )

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            test_loss = criterion(
                outputs,
                labels
            )

            test_loss_sum += (
                test_loss.item() *
                input_ids.size(0)
            )

            test_sample_count += (
                input_ids.size(0)
            )

    average_test_loss = (
        test_loss_sum /
        test_sample_count
    )

    if average_test_loss < best_val_loss:

        best_val_loss = average_test_loss
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "best_model_gpt2_text.pth"
        )

    if (epoch + 1) % 5 == 0:

        print(
            f"Epoch {epoch + 1:03d}, "
            f"Train={average_train_loss:.4f}, "
            f"Test={average_test_loss:.4f}, "
            f"Best Epoch={best_epoch}"
        )


print("\nBest Test Loss:", best_val_loss)
print("Best Epoch:", best_epoch)


# ============================================================
# Load best model
# ============================================================

checkpoint = torch.load(
    "best_model_gpt2_text.pth",
    map_location=device
)

model.load_state_dict(checkpoint)
model.eval()


# ============================================================
# Inference
# ============================================================

all_probabilities = []
all_predictions = []
all_targets = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )

        labels = batch["label"]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        all_probabilities.append(
            probabilities.cpu()
        )

        all_predictions.append(
            predictions.cpu()
        )

        all_targets.append(
            labels.cpu()

                    )


probabilities = torch.cat(
    all_probabilities,
    dim=0
).numpy()

predicted = torch.cat(
    all_predictions,
    dim=0
).numpy()

y_true = torch.cat(
    all_targets,
    dim=0
).numpy()


# ============================================================
# Evaluation
# ============================================================

results = classification_report(
    y_true,
    predicted,
    digits=4,
    zero_division=0
)

print("\nClassification report:")
print(results)

bal_acc = balanced_accuracy_score(
    y_true,
    predicted
)

f1 = f1_score(
    y_true,
    predicted,
    zero_division=0
)

auc = roc_auc_score(
    y_true,
    probabilities[:, 1]
)

print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"AUC:               {auc:.4f}")

Downloading...
From: https://drive.google.com/uc?id=1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q
To: /content/clinical_data.json
100%|██████████| 810k/810k [00:00<00:00, 9.97MB/s]
Downloading...
From: https://drive.google.com/uc?id=1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F
To: /content/pathological_data.json
100%|██████████| 458k/458k [00:00<00:00, 2.57MB/s]


(763, 49)
Clinical shape: (763, 32)
Pathology shape: (763, 18)
Patients: 332
0    191
1    141
Name: count, dtype: int64
Using device: cuda

Train class distribution:
0    152
1    113
Name: count, dtype: int64

Test class distribution:
0    39
1    28
Name: count, dtype: int64


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Class weights: tensor([0.8717, 1.1726], device='cuda:0')
Trainable parameters: 100226
Epoch 005, Train=0.7419, Test=0.7081, Best Epoch=5
Epoch 010, Train=0.6947, Test=0.6952, Best Epoch=9
Epoch 015, Train=0.6934, Test=0.6956, Best Epoch=9
Epoch 020, Train=0.6977, Test=0.6926, Best Epoch=19
Epoch 025, Train=0.6892, Test=0.6934, Best Epoch=22
Epoch 030, Train=0.6964, Test=0.6907, Best Epoch=30
Epoch 035, Train=0.6947, Test=0.6969, Best Epoch=30
Epoch 040, Train=0.6953, Test=0.6907, Best Epoch=38
Epoch 045, Train=0.6890, Test=0.6953, Best Epoch=41
Epoch 050, Train=0.6854, Test=0.6927, Best Epoch=48
Epoch 055, Train=0.6924, Test=0.6892, Best Epoch=55
Epoch 060, Train=0.6901, Test=0.6897, Best Epoch=55
Epoch 065, Train=0.6929, Test=0.6939, Best Epoch=55
Epoch 070, Train=0.6961, Test=0.6916, Best Epoch=67
Epoch 075, Train=0.6927, Test=0.6907, Best Epoch=67
Epoch 080, Train=0.6921, Test=0.6913, Best Epoch=78
Epoch 085, Train=0.6891, Test=0.6875, Best Epoch=85
Epoch 090, Train=0.6917, Test=0.

In [ ]:
MAX_TOKEN = 128, epoch 100
Balanced Accuracy: 0.7239
F1-score:          0.6786
AUC:               0.8489

MAX_TOKEN = 256, epoch 100:
Balanced Accuracy: 0.6937
F1-score:          0.5909
AUC:               0.8434

MAX_TOKEN = 256, epoch 200:
Balanced Accuracy: 0.7546
F1-score:          0.7143
AUC:               0.8507

MAX_TOKEN = 256, epoch 300:
Balanced Accuracy: 0.6955
F1-score:          0.6667
AUC:               0.8480


Narrative GPT2 with SMOTE

In [ ]:
# ============================================================
# GPT-2 Narrative Classification with SMOTENC
# HPV Classification
#
# Pipeline:
# Tabular data
#   -> Train/Test split
#   -> Missing-value imputation
#   -> Encode categorical features
#   -> SMOTENC on TRAINING DATA ONLY
#   -> Decode categorical features
#   -> Convert patient records to narrative text
#   -> GPT-2 tokenizer
#   -> GPT-2 classifier
# ============================================================


# ============================================================
# Imports
# ============================================================

import os
import random
import json

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import OrdinalEncoder

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

from imblearn.over_sampling import SMOTENC

from transformers import (
    GPT2Tokenizer,
    GPT2Model
)


# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):

    os.environ["PYTHONHASHSEED"] = str(seed)

    os.environ[
        "CUBLAS_WORKSPACE_CONFIG"
    ] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


SEED = 42

seed_everything(SEED)


# ============================================================
# Device
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", device)


# ============================================================
# Read Data
# ============================================================

import gdown


# Clinical data
gdown.download(
    id="1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q",
    output="clinical_data.json",
    quiet=False
)


# Pathological data
gdown.download(
    id="1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F",
    output="pathological_data.json",
    quiet=False
)


with open(
    "clinical_data.json",
    "r"
) as f:

    clinical = pd.DataFrame(
        json.load(f)
    )


with open(
    "pathological_data.json",
    "r"
) as f:

    pathology = pd.DataFrame(
        json.load(f)
    )


df = clinical.merge(
    pathology,
    on="patient_id",
    how="inner"
)


print("Merged shape:", df.shape)

print(
    "Clinical shape:",
    clinical.shape
)

print(
    "Pathology shape:",
    pathology.shape
)


# ============================================================
# HPV Labels
# ============================================================

df = df[
    df[
        "hpv_association_p16"
    ].isin(
        [
            "positive",
            "negative"
        ]
    )
].copy()


df["HPV"] = df[
    "hpv_association_p16"
].map(
    {
        "negative": 0,
        "positive": 1
    }
)


print(
    "\nOriginal HPV distribution:"
)

print(
    df["HPV"].value_counts()
)


# ============================================================
# Hancock Features
# ============================================================

features = [

    "age_at_initial_diagnosis",

    "sex",

    "smoking_status",

    "primarily_metastasis",

    "first_treatment_intent",

    "first_treatment_modality",

    "days_to_first_treatment",

    "adjuvant_treatment_intent",

    "adjuvant_radiotherapy",

    "adjuvant_radiotherapy_modality",

    "adjuvant_systemic_therapy",

    "adjuvant_systemic_therapy_modality",

    "adjuvant_radiochemotherapy",

    "primary_tumor_site",

    "pT_stage",

    "pN_stage",

    "histologic_type",

    "number_of_positive_lymph_nodes",

    "number_of_resected_lymph_nodes",

    "perinodal_invasion",

    "lymphovascular_invasion_L",

    "vascular_invasion_V",

    "perineural_invasion_Pn",

    "resection_status",

    "infiltration_depth_in_mm"
]


# ============================================================
# Categorical / Numerical Features
# ============================================================

categorical_features = [

    "sex",

    "smoking_status",

    "primarily_metastasis",

    "first_treatment_intent",

    "first_treatment_modality",

    "adjuvant_treatment_intent",

    "adjuvant_radiotherapy",

    "adjuvant_radiotherapy_modality",

    "adjuvant_systemic_therapy",

    "adjuvant_systemic_therapy_modality",

    "adjuvant_radiochemotherapy",

    "primary_tumor_site",

    "pT_stage",

    "pN_stage",

    "histologic_type",

    "perinodal_invasion",

    "lymphovascular_invasion_L",

    "vascular_invasion_V",

    "perineural_invasion_Pn",

    "resection_status"
]


numerical_features = [

    "age_at_initial_diagnosis",

    "days_to_first_treatment",

    "number_of_positive_lymph_nodes",

    "number_of_resected_lymph_nodes",

    "infiltration_depth_in_mm"
]


# ============================================================
# Features / Target
# ============================================================

X = df[
    features
].copy()

y = df[
    "HPV"
].copy()


# ============================================================
# Train-Test Split
#
# IMPORTANT:
# Split BEFORE imputation and SMOTE.
# ============================================================

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=SEED,
        stratify=y
    )
)


# Reset index
X_train = X_train.reset_index(
    drop=True
)

X_test = X_test.reset_index(
    drop=True
)

y_train = y_train.reset_index(
    drop=True
)

y_test = y_test.reset_index(
    drop=True
)


print(
    "\nTrain class distribution "
    "before SMOTE:"
)

print(
    y_train.value_counts()
)


print(
    "\nTest class distribution:"
)

print(
    y_test.value_counts()
)


# ============================================================
# Missing Values
#
# Calculate imputation statistics using
# TRAINING DATA ONLY.
# ============================================================

for col in categorical_features:

    mode_value = (
        X_train[col]
        .mode(dropna=True)
        .iloc[0]
    )

    X_train[col] = (
        X_train[col]
        .fillna(mode_value)
    )

    X_test[col] = (
        X_test[col]
        .fillna(mode_value)
    )


for col in numerical_features:

    median_value = (
        X_train[col]
        .median()
    )

    X_train[col] = (
        X_train[col]
        .fillna(median_value)
    )

    X_test[col] = (
        X_test[col]
        .fillna(median_value)
    )


# ============================================================
# Encode Categorical Features
#
# SMOTENC needs categorical variables represented
# numerically.
#
# We will later convert them back to their
# original category names before generating text.
# ============================================================

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)


X_train_encoded = (
    X_train.copy()
)


X_train_encoded[
    categorical_features
] = encoder.fit_transform(
    X_train[
        categorical_features
    ]
)


# ============================================================
# Categorical Feature Indices for SMOTENC
# ============================================================

categorical_indices = [

    X_train_encoded.columns.get_loc(
        col
    )

    for col in categorical_features
]


print(
    "\nCategorical feature indices:"
)

print(
    categorical_indices
)


# ============================================================
# SMOTENC
#
# Apply ONLY to training set.
#
# SMOTENC is used instead of normal SMOTE because
# the data contain both continuous and categorical
# variables.
# ============================================================

smote = SMOTENC(

    categorical_features=(
        categorical_indices
    ),

    random_state=SEED
)


X_train_smote, y_train_smote = (
    smote.fit_resample(
        X_train_encoded,
        y_train
    )
)


# Ensure DataFrame
X_train_smote = pd.DataFrame(
    X_train_smote,
    columns=features
)


y_train_smote = pd.Series(
    y_train_smote,
    name="HPV"
)


print(
    "\nTrain class distribution "
    "after SMOTENC:"
)

print(
    y_train_smote.value_counts()
)


print(
    "\nNumber of original "
    "training patients:",
    len(X_train)
)


print(
    "Number of training patients "
    "after SMOTENC:",
    len(X_train_smote)
)


# ============================================================
# Convert Encoded Categorical Features Back to Text
# ============================================================

decoded_categories = (
    encoder.inverse_transform(
        X_train_smote[
            categorical_features
        ]
    )
)


X_train_smote[
    categorical_features
] = decoded_categories


# ============================================================
# Clean Numerical Synthetic Values
#
# SMOTE interpolates continuous features.
#
# Some clinical variables represent integer counts.
# Round these variables to clinically sensible values.
# ============================================================

integer_features = [

    "age_at_initial_diagnosis",

    "days_to_first_treatment",

    "number_of_positive_lymph_nodes",

    "number_of_resected_lymph_nodes"
]


for col in integer_features:

    X_train_smote[col] = (
        X_train_smote[col]
        .round()
        .astype(int)
    )


# Counts cannot be negative
count_features = [

    "days_to_first_treatment",

    "number_of_positive_lymph_nodes",

    "number_of_resected_lymph_nodes"
]


for col in count_features:

    X_train_smote[col] = (
        X_train_smote[col]
        .clip(lower=0)
    )


# Infiltration depth should not be negative
X_train_smote[
    "infiltration_depth_in_mm"
] = (

    X_train_smote[
        "infiltration_depth_in_mm"
    ]
    .clip(lower=0)
    .round(2)
)


# ============================================================
# Narrative Function
#
# Same idea as original narrative GPT-2 implementation.
# ============================================================

def patient_to_text(row):

    text = (

        f"This patient was diagnosed at age "
        f"{row['age_at_initial_diagnosis']}. "

        f"Sex is {row['sex']}. "

        f"Smoking status is "
        f"{row['smoking_status']}. "

        f"Primary tumour site is "
        f"{row['primary_tumor_site']}. "

        f"Histologic type is "
        f"{row['histologic_type']}. "

        f"Pathological T stage is "
        f"{row['pT_stage']}. "

        f"Pathological N stage is "
        f"{row['pN_stage']}. "

        f"The patient has "
        f"{row['number_of_positive_lymph_nodes']} "
        f"positive lymph nodes and "

        f"{row['number_of_resected_lymph_nodes']} "
        f"resected lymph nodes. "

        f"Perinodal invasion status is "
        f"{row['perinodal_invasion']}. "

        f"Lymphovascular invasion status is "
        f"{row['lymphovascular_invasion_L']}. "

        f"Vascular invasion status is "
        f"{row['vascular_invasion_V']}. "

        f"Perineural invasion status is "
        f"{row['perineural_invasion_Pn']}. "

        f"Resection status is "
        f"{row['resection_status']}. "

        f"Infiltration depth is "
        f"{row['infiltration_depth_in_mm']} mm. "

        f"First treatment intent was "
        f"{row['first_treatment_intent']}. "

        f"First treatment modality was "
        f"{row['first_treatment_modality']}. "

        f"Days to first treatment was "
        f"{row['days_to_first_treatment']}. "

        f"Adjuvant treatment intent was "
        f"{row['adjuvant_treatment_intent']}. "

        f"Adjuvant radiotherapy was "
        f"{row['adjuvant_radiotherapy']}. "

        f"Adjuvant radiotherapy modality was "
        f"{row['adjuvant_radiotherapy_modality']}. "

        f"Adjuvant systemic therapy was "
        f"{row['adjuvant_systemic_therapy']}. "

        f"Adjuvant systemic therapy modality was "
        f"{row['adjuvant_systemic_therapy_modality']}. "

        f"Adjuvant radiochemotherapy was "
        f"{row['adjuvant_radiochemotherapy']}. "

        f"Predict whether the patient is "
        f"HPV positive or HPV negative."
    )

    return text


# ============================================================
# Create Training Narratives
#
# IMPORTANT:
# These include the synthetic SMOTENC patients.
# ============================================================

train_texts = (

    X_train_smote
    .apply(
        patient_to_text,
        axis=1
    )
    .tolist()
)


# ============================================================
# Create Test Narratives
#
# Test data remain completely untouched by SMOTE.
# ============================================================

test_texts = (

    X_test
    .apply(
        patient_to_text,
        axis=1
    )
    .tolist()
)


train_labels = (
    y_train_smote
    .astype(int)
    .tolist()
)


test_labels = (
    y_test
    .astype(int)
    .tolist()
)


print(
    "\nTraining narratives:",
    len(train_texts)
)

print(
    "Test narratives:",
    len(test_texts)
)


print(
    "\nTraining labels:"
)

print(
    pd.Series(
        train_labels
    ).value_counts()
)


# ============================================================
# Example Original / Synthetic Narrative
# ============================================================

print(
    "\nExample training narrative:\n"
)

print(
    train_texts[0]
)


# ============================================================
# GPT-2 Tokenizer
# ============================================================

MODEL_NAME = (
    "openai-community/gpt2"
)


tokenizer = (
    GPT2Tokenizer
    .from_pretrained(
        MODEL_NAME
    )
)


# GPT-2 has no padding token
tokenizer.pad_token = (
    tokenizer.eos_token
)


MAX_LENGTH = 256


# ============================================================
# Dataset
# ============================================================

class HPVTextDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length=256
    ):

        self.texts = texts

        self.labels = labels

        self.tokenizer = tokenizer

        self.max_length = (
            max_length
        )


    def __len__(self):

        return len(
            self.texts
        )


    def __getitem__(
        self,
        index
    ):

        encoding = (
            self.tokenizer(
                self.texts[index],

                max_length=(
                    self.max_length
                ),

                padding="max_length",

                truncation=True,

                return_tensors="pt"
            )
        )


        return {

            "input_ids":

                encoding[
                    "input_ids"
                ].squeeze(0),

            "attention_mask":

                encoding[
                    "attention_mask"
                ].squeeze(0),

            "label":

                torch.tensor(
                    self.labels[index],
                    dtype=torch.long
                )
        }


# ============================================================
# Datasets
# ============================================================

train_dataset = HPVTextDataset(

    texts=train_texts,

    labels=train_labels,

    tokenizer=tokenizer,

    max_length=MAX_LENGTH
)


test_dataset = HPVTextDataset(

    texts=test_texts,

    labels=test_labels,

    tokenizer=tokenizer,

    max_length=MAX_LENGTH
)


# ============================================================
# DataLoaders
# ============================================================

BATCH_SIZE = 256


train_generator = (
    torch.Generator()
)

train_generator.manual_seed(
    SEED
)


train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    generator=train_generator,

    num_workers=0,

    pin_memory=(
        torch.cuda.is_available()
    )
)


test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=(
        torch.cuda.is_available()
    )
)


# ============================================================
# GPT-2 Text Classifier
# ============================================================

class HPVNetGPT2Text(nn.Module):

    def __init__(
        self,

        model_name=(
            "openai-community/gpt2"
        ),

        num_classes=2,

        dropout=0.1,

        freeze_gpt2=True
    ):

        super().__init__()


        self.freeze_gpt2 = (
            freeze_gpt2
        )


        self.gpt2 = (
            GPT2Model
            .from_pretrained(
                model_name
            )
        )


        self.gpt2.config.use_cache = (
            False
        )


        self.gpt2.config.pad_token_id = (
            tokenizer.pad_token_id
        )


        hidden_size = (
            self.gpt2
            .config
            .hidden_size
        )


        self.classifier = nn.Sequential(

            nn.LayerNorm(
                hidden_size
            ),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_size,
                128
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                128,
                num_classes
            )
        )


        if freeze_gpt2:

            self.gpt2.requires_grad_(
                False
            )


    def train(
        self,
        mode=True
    ):

        super().train(
            mode
        )


        # Keep frozen GPT-2 in evaluation mode
        if self.freeze_gpt2:

            self.gpt2.eval()


        return self


    def forward(
        self,
        input_ids,
        attention_mask
    ):

        outputs = self.gpt2(

            input_ids=input_ids,

            attention_mask=(
                attention_mask
            ),

            use_cache=False,

            return_dict=True
        )


        hidden_states = (
            outputs
            .last_hidden_state
        )


        # ====================================================
        # Masked Mean Pooling
        # ====================================================

        expanded_mask = (

            attention_mask

            .unsqueeze(-1)

            .expand_as(
                hidden_states
            )

            .float()
        )


        hidden_sum = (

            hidden_states
            *
            expanded_mask

        ).sum(
            dim=1
        )


        valid_token_count = (

            expanded_mask

            .sum(
                dim=1
            )

            .clamp(
                min=1e-9
            )
        )


        pooled_output = (

            hidden_sum
            /
            valid_token_count
        )


        logits = (
            self.classifier(
                pooled_output
            )
        )


        return logits


# ============================================================
# Model
# ============================================================

model = HPVNetGPT2Text(

    model_name=MODEL_NAME,

    num_classes=2,

    dropout=0.1,

    freeze_gpt2=True

).to(
    device
)


# ============================================================
# Loss
#
# IMPORTANT:
# No class weights because SMOTE has already balanced
# the training set.
# ============================================================

criterion = (
    nn.CrossEntropyLoss()
)


# ============================================================
# Optimiser
# ============================================================

optimizer = (
    torch.optim.AdamW(

        filter(

            lambda parameter:
                parameter.requires_grad,

            model.parameters()
        ),

        lr=1e-3,

        weight_decay=1e-4
    )
)


print(
    "\nTrainable parameters:",
    sum(
        parameter.numel()

        for parameter
        in model.parameters()

        if parameter.requires_grad
    )
)


# ============================================================
# Training
# ============================================================

EPOCHS = 200


best_test_loss = float(
    "inf"
)

best_epoch = 0


for epoch in range(
    EPOCHS
):

    # ========================================================
    # Training
    # ========================================================

    model.train()


    train_loss_sum = 0.0

    train_sample_count = 0


    for batch in train_loader:

        input_ids = (

            batch[
                "input_ids"
            ]

            .to(
                device,
                non_blocking=True
            )
        )


        attention_mask = (

            batch[
                "attention_mask"
            ]

            .to(
                device,
                non_blocking=True
            )
        )


        labels = (

            batch[
                "label"
            ]

            .to(
                device,
                non_blocking=True
            )
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        outputs = model(

            input_ids=input_ids,

            attention_mask=(
                attention_mask
            )
        )


        train_loss = criterion(

            outputs,

            labels
        )


        train_loss.backward()


        torch.nn.utils.clip_grad_norm_(

            filter(

                lambda parameter:
                    parameter.requires_grad,

                model.parameters()
            ),

            max_norm=1.0
        )


        optimizer.step()


        train_loss_sum += (

            train_loss.item()
            *
            input_ids.size(0)
        )


        train_sample_count += (
            input_ids.size(0)
        )


    average_train_loss = (

        train_loss_sum
        /
        train_sample_count
    )


    # ========================================================
    # Test
    # ========================================================

    model.eval()


    test_loss_sum = 0.0

    test_sample_count = 0


    with torch.no_grad():

        for batch in test_loader:

            input_ids = (

                batch[
                    "input_ids"
                ]

                .to(
                    device,
                    non_blocking=True
                )
            )


            attention_mask = (

                batch[
                    "attention_mask"
                ]

                .to(
                    device,
                    non_blocking=True
                )
            )


            labels = (

                batch[
                    "label"
                ]

                .to(
                    device,
                    non_blocking=True
                )
            )


            outputs = model(

                input_ids=input_ids,

                attention_mask=(
                    attention_mask
                )
            )


            test_loss = criterion(

                outputs,

                labels
            )


            test_loss_sum += (

                test_loss.item()
                *
                input_ids.size(0)
            )


            test_sample_count += (
                input_ids.size(0)
            )


    average_test_loss = (

        test_loss_sum
        /
        test_sample_count
    )


    # ========================================================
    # Save Best Model
    # ========================================================

    if (
        average_test_loss
        <
        best_test_loss
    ):

        best_test_loss = (
            average_test_loss
        )

        best_epoch = (
            epoch + 1
        )


        torch.save(

            model.state_dict(),

            "best_model_gpt2_text_smote.pth"
        )


    if (
        (epoch + 1) % 5
        ==
        0
    ):

        print(

            f"Epoch {epoch + 1:03d}, "

            f"Train={average_train_loss:.4f}, "

            f"Test={average_test_loss:.4f}, "

            f"Best Epoch={best_epoch}"
        )


print(
    "\nBest Test Loss:",
    best_test_loss
)

print(
    "Best Epoch:",
    best_epoch
)


# ============================================================
# Load Best Model
# ============================================================

checkpoint = torch.load(

    "best_model_gpt2_text_smote.pth",

    map_location=device
)


model.load_state_dict(
    checkpoint
)

model.eval()


# ============================================================
# Inference
# ============================================================

all_probabilities = []

all_predictions = []

all_targets = []


with torch.no_grad():

    for batch in test_loader:

        input_ids = (

            batch[
                "input_ids"
            ]

            .to(
                device,
                non_blocking=True
            )
        )


        attention_mask = (

            batch[
                "attention_mask"
            ]

            .to(
                device,
                non_blocking=True
            )
        )


        labels = (
            batch[
                "label"
            ]
        )


        outputs = model(

            input_ids=input_ids,

            attention_mask=(
                attention_mask
            )
        )


        probabilities = (
            torch.softmax(
                outputs,
                dim=1
            )
        )


        predictions = (
            torch.argmax(
                probabilities,
                dim=1
            )
        )


        all_probabilities.append(
            probabilities.cpu()
        )


        all_predictions.append(
            predictions.cpu()
        )


        all_targets.append(
            labels.cpu()
        )


# ============================================================
# Concatenate Predictions
# ============================================================

probabilities = (

    torch.cat(
        all_probabilities,
        dim=0
    )

    .numpy()
)


predicted = (

    torch.cat(
        all_predictions,
        dim=0
    )

    .numpy()
)


y_true = (

    torch.cat(
        all_targets,
        dim=0
    )

    .numpy()
)


# ============================================================
# Evaluation
# ============================================================

print(
    "\nClassification Report:\n"
)


print(
    classification_report(

        y_true,

        predicted,

        digits=4,

        zero_division=0
    )
)


# ============================================================
# Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_true,
    predicted
)


print(
    "\nConfusion Matrix:"
)

print(
    cm
)


# ============================================================
# Metrics
# ============================================================

accuracy = accuracy_score(
    y_true,
    predicted
)


balanced_accuracy = (
    balanced_accuracy_score(
        y_true,
        predicted
    )
)


f1 = f1_score(

    y_true,

    predicted,

    zero_division=0
)


auc = roc_auc_score(

    y_true,

    probabilities[:, 1]
)


print()

print(
    f"Accuracy:          "
    f"{accuracy:.4f}"
)

print(
    f"Balanced Accuracy: "
    f"{balanced_accuracy:.4f}"
)

print(
    f"F1-score:          "
    f"{f1:.4f}"
)

print(
    f"AUC:               "
    f"{auc:.4f}"
)


# ============================================================
# Summary
# ============================================================

print()

print(
    "========================================"
)

print(
    "GPT2 TEXT + SMOTENC SUMMARY"
)

print(
    "========================================"
)

print(
    "Model             : GPT-2 Narrative"
)

print(
    "Input             : Clinical narrative"
)

print(
    "Features          :",
    len(features)
)

print(
    "Balancing         : SMOTENC"
)

print(
    "Class weights     : None"
)

print(
    "GPT-2 frozen      : Yes"
)

print(
    "Learning Rate     : 0.001"
)

print(
    "Optimiser         : AdamW"
)

print(
    "Epochs            :",
    EPOCHS
)

print(
    "Best Epoch        :",
    best_epoch
)

print(
    "Best Test Loss    :",
    round(
        best_test_loss,
        4
    )
)

print(
    "Original Train N  :",
    len(X_train)
)

print(
    "SMOTE Train N     :",
    len(train_dataset)
)

print(
    "Test N            :",
    len(test_dataset)
)

print(
    "Seed              :",
    SEED
)

Using device: cuda


Downloading...
From: https://drive.google.com/uc?id=1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q
To: /content/clinical_data.json
100%|██████████| 810k/810k [00:00<00:00, 7.17MB/s]
Downloading...
From: https://drive.google.com/uc?id=1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F
To: /content/pathological_data.json
100%|██████████| 458k/458k [00:00<00:00, 6.49MB/s]


Merged shape: (763, 49)
Clinical shape: (763, 32)
Pathology shape: (763, 18)

Original HPV distribution:
HPV
0    191
1    141
Name: count, dtype: int64

Train class distribution before SMOTE:
HPV
0    152
1    113
Name: count, dtype: int64

Test class distribution:
HPV
0    39
1    28
Name: count, dtype: int64

Categorical feature indices:
[1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 19, 20, 21, 22, 23]

Train class distribution after SMOTENC:
HPV
0    152
1    152
Name: count, dtype: int64

Number of original training patients: 265
Number of training patients after SMOTENC: 304

Training narratives: 304
Test narratives: 67

Training labels:
0    152
1    152
Name: count, dtype: int64

Example training narrative:

This patient was diagnosed at age 71. Sex is male. Smoking status is non-smoker. Primary tumour site is Oropharynx. Histologic type is SCC_Basaloid. Pathological T stage is pT3. Pathological N stage is pN1. The patient has 1 positive lymph nodes and 45 resected lymph

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Trainable parameters: 100226
Epoch 005, Train=0.6897, Test=0.7266, Best Epoch=3
Epoch 010, Train=0.6967, Test=0.6788, Best Epoch=10
Epoch 015, Train=0.6989, Test=0.7204, Best Epoch=11
Epoch 020, Train=0.6959, Test=0.6851, Best Epoch=11
Epoch 025, Train=0.6918, Test=0.6879, Best Epoch=11
Epoch 030, Train=0.6908, Test=0.7040, Best Epoch=27
Epoch 035, Train=0.6912, Test=0.6843, Best Epoch=27
Epoch 040, Train=0.6884, Test=0.6848, Best Epoch=27
Epoch 045, Train=0.6875, Test=0.6889, Best Epoch=27
Epoch 050, Train=0.6899, Test=0.6858, Best Epoch=27
Epoch 055, Train=0.6882, Test=0.6859, Best Epoch=52
Epoch 060, Train=0.6865, Test=0.6855, Best Epoch=52
Epoch 065, Train=0.6833, Test=0.6991, Best Epoch=52
Epoch 070, Train=0.6841, Test=0.6716, Best Epoch=70
Epoch 075, Train=0.6855, Test=0.6781, Best Epoch=70
Epoch 080, Train=0.6830, Test=0.6812, Best Epoch=70
Epoch 085, Train=0.6908, Test=0.6779, Best Epoch=84
Epoch 090, Train=0.6824, Test=0.6620, Best Epoch=90
Epoch 095, Train=0.6766, Test=0.666

#Sol2 RandomOverSampler without weighted loss

In [ ]:
# ============================================================
# Imports
# ============================================================

import os
import random
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import gdown

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, balanced_accuracy_score, f1_score, roc_auc_score
from imblearn.over_sampling import RandomOverSampler
from transformers import GPT2Tokenizer, GPT2Model


# ============================================================
# Read Data
# ============================================================

gdown.download(
    id="1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q",
    output="clinical_data.json",
    quiet=False
)

gdown.download(
    id="1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F",
    output="pathological_data.json",
    quiet=False
)

with open("clinical_data.json", "r") as f:
    clinical = pd.DataFrame(json.load(f))

with open("pathological_data.json", "r") as f:
    pathology = pd.DataFrame(json.load(f))

df = clinical.merge(pathology, on="patient_id", how="inner")

print("Merged shape:", df.shape)
print("Clinical shape:", clinical.shape)
print("Pathology shape:", pathology.shape)


# ============================================================
# HPV Labels
# ============================================================

df = df[df["hpv_association_p16"].isin(["positive", "negative"])].copy()
df["HPV"] = df["hpv_association_p16"].map({"negative": 0, "positive": 1})

print("\nHPV distribution:")
print(df["HPV"].value_counts())


# ============================================================
# GPT2 Hancock Features
# ============================================================

features = [
    "age_at_initial_diagnosis",
    "sex",
    "smoking_status",
    "primarily_metastasis",
    "first_treatment_intent",
    "first_treatment_modality",
    "days_to_first_treatment",
    "adjuvant_treatment_intent",
    "adjuvant_radiotherapy",
    "adjuvant_radiotherapy_modality",
    "adjuvant_systemic_therapy",
    "adjuvant_systemic_therapy_modality",
    "adjuvant_radiochemotherapy",
    "primary_tumor_site",
    "pT_stage",
    "pN_stage",
    "histologic_type",
    "number_of_positive_lymph_nodes",
    "number_of_resected_lymph_nodes",
    "perinodal_invasion",
    "lymphovascular_invasion_L",
    "vascular_invasion_V",
    "perineural_invasion_Pn",
    "resection_status",
    "infiltration_depth_in_mm"
]


# ============================================================
# Missing Values
# ============================================================

for col in features:
    if df[col].dtype == object:
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())


# ============================================================
# Narrative Function
# ============================================================

def patient_to_text(row):
    text = (
        f"This patient was diagnosed at age {row['age_at_initial_diagnosis']}. "
        f"Sex is {row['sex']}. "
        f"Smoking status is {row['smoking_status']}. "
        f"Primary tumour site is {row['primary_tumor_site']}. "
        f"Histologic type is {row['histologic_type']}. "
        f"Pathological T stage is {row['pT_stage']}. "
        f"Pathological N stage is {row['pN_stage']}. "
        f"The patient has {row['number_of_positive_lymph_nodes']} positive lymph nodes and "
        f"{row['number_of_resected_lymph_nodes']} resected lymph nodes. "
        f"Perinodal invasion status is {row['perinodal_invasion']}. "
        f"Lymphovascular invasion status is {row['lymphovascular_invasion_L']}. "
        f"Vascular invasion status is {row['vascular_invasion_V']}. "
        f"Perineural invasion status is {row['perineural_invasion_Pn']}. "
        f"Resection status is {row['resection_status']}. "
        f"Infiltration depth is {row['infiltration_depth_in_mm']} mm. "
        f"First treatment intent was {row['first_treatment_intent']}. "
        f"First treatment modality was {row['first_treatment_modality']}. "
        f"Days to first treatment was {row['days_to_first_treatment']}. "
        f"Adjuvant treatment intent was {row['adjuvant_treatment_intent']}. "
        f"Adjuvant radiotherapy was {row['adjuvant_radiotherapy']}. "
        f"Adjuvant radiotherapy modality was {row['adjuvant_radiotherapy_modality']}. "
        f"Adjuvant systemic therapy was {row['adjuvant_systemic_therapy']}. "
        f"Adjuvant systemic therapy modality was {row['adjuvant_systemic_therapy_modality']}. "
        f"Adjuvant radiochemotherapy was {row['adjuvant_radiochemotherapy']}. "
        f"Predict whether the patient is HPV positive or HPV negative."
    )
    return text


# ============================================================
# Create GPT2 Text Corpus
# ============================================================

texts = df.apply(patient_to_text, axis=1).tolist()
labels = df["HPV"].tolist()

print("\nPatients:", len(texts))
print("\nOriginal class distribution:")
print(pd.Series(labels).value_counts())


# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


SEED = 42
seed_everything(SEED)


# ============================================================
# Device
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nUsing device:", device)


# ============================================================
# Train Test Split
# ============================================================

train_texts, test_texts, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.20,
    random_state=SEED,
    stratify=labels
)

print("\nTrain class distribution BEFORE oversampling:")
print(pd.Series(y_train).value_counts())

print("\nTest class distribution:")
print(pd.Series(y_test).value_counts())


# ============================================================
# Random Oversampling
# Training Data Only
# ============================================================

ros = RandomOverSampler(random_state=SEED)

train_indices = np.arange(len(train_texts)).reshape(-1, 1)
train_indices_resampled, y_train_resampled = ros.fit_resample(train_indices, y_train)

train_texts = [train_texts[index[0]] for index in train_indices_resampled]
y_train = y_train_resampled.tolist()

print("\nTrain class distribution AFTER oversampling:")
print(pd.Series(y_train).value_counts())

print("\nNumber of training samples after oversampling:", len(train_texts))


# ============================================================
# Tokenizer
# ============================================================

MODEL_NAME = "openai-community/gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

MAX_LENGTH = 256


# ============================================================
# Dataset
# ============================================================

class HPVTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoding = self.tokenizer(
            self.texts[index],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[index], dtype=torch.long)
        }


# ============================================================
# Datasets
# ============================================================

train_dataset = HPVTextDataset(
    texts=train_texts,
    labels=y_train,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

test_dataset = HPVTextDataset(
    texts=test_texts,
    labels=y_test,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

print("\nTraining dataset size:", len(train_dataset))
print("Test dataset size:", len(test_dataset))


# ============================================================
# DataLoaders
# ============================================================

BATCH_SIZE = 256

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


# ============================================================
# GPT2 Text Classifier
# ============================================================

class HPVNetGPT2Text(nn.Module):
    def __init__(
        self,
        model_name="openai-community/gpt2",
        num_classes=2,
        dropout=0.1,
        freeze_gpt2=True
    ):
        super().__init__()

        self.freeze_gpt2 = freeze_gpt2
        self.gpt2 = GPT2Model.from_pretrained(model_name)

        self.gpt2.config.use_cache = False
        self.gpt2.config.pad_token_id = tokenizer.pad_token_id

        hidden_size = self.gpt2.config.hidden_size

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

        if freeze_gpt2:
            self.gpt2.requires_grad_(False)

    def train(self, mode=True):
        super().train(mode)

        if self.freeze_gpt2:
            self.gpt2.eval()

        return self

    def forward(self, input_ids, attention_mask):
        outputs = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True
        )

        hidden_states = outputs.last_hidden_state

        # Masked mean pooling
        expanded_mask = attention_mask.unsqueeze(-1).expand_as(hidden_states).float()
        hidden_sum = (hidden_states * expanded_mask).sum(dim=1)
        valid_token_count = expanded_mask.sum(dim=1).clamp(min=1e-9)
        pooled_output = hidden_sum / valid_token_count

        logits = self.classifier(pooled_output)
        return logits


# ============================================================
# Model
# ============================================================

model = HPVNetGPT2Text(
    model_name=MODEL_NAME,
    num_classes=2,
    dropout=0.1,
    freeze_gpt2=True
).to(device)


# ============================================================
# Loss
# ============================================================

criterion = nn.CrossEntropyLoss()


# ============================================================
# Optimiser
# ============================================================

optimizer = torch.optim.AdamW(
    filter(lambda parameter: parameter.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)

print(
    "\nTrainable parameters:",
    sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
)


# ============================================================
# Training
# ============================================================

EPOCHS = 200

best_val_loss = float("inf")
best_epoch = 0

for epoch in range(EPOCHS):

    # Training
    model.train()

    train_loss_sum = 0.0
    train_sample_count = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels_batch = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        train_loss = criterion(outputs, labels_batch)
        train_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            filter(lambda parameter: parameter.requires_grad, model.parameters()),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum += train_loss.item() * input_ids.size(0)
        train_sample_count += input_ids.size(0)

    average_train_loss = train_loss_sum / train_sample_count


    # Validation / Test
    model.eval()

    test_loss_sum = 0.0
    test_sample_count = 0

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels_batch = batch["label"].to(device, non_blocking=True)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            test_loss = criterion(outputs, labels_batch)

            test_loss_sum += test_loss.item() * input_ids.size(0)
            test_sample_count += input_ids.size(0)

    average_test_loss = test_loss_sum / test_sample_count


    # Save Best Model
    if average_test_loss < best_val_loss:
        best_val_loss = average_test_loss
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "best_model_gpt2_text_oversampling.pth"
        )


    # Progress
    if (epoch + 1) % 5 == 0:
        print(
            f"Epoch {epoch + 1:03d}, "
            f"Train={average_train_loss:.4f}, "
            f"Test={average_test_loss:.4f}, "
            f"Best Epoch={best_epoch}"
        )


print("\nBest Test Loss:", best_val_loss)
print("Best Epoch:", best_epoch)


# ============================================================
# Load Best Model
# ============================================================

checkpoint = torch.load(
    "best_model_gpt2_text_oversampling.pth",
    map_location=device
)

model.load_state_dict(checkpoint)
model.eval()


# ============================================================
# Inference
# ============================================================

all_probabilities = []
all_predictions = []
all_targets = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels_batch = batch["label"]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probabilities = torch.softmax(outputs, dim=1)
        predictions = torch.argmax(probabilities, dim=1)

        all_probabilities.append(probabilities.cpu())
        all_predictions.append(predictions.cpu())
        all_targets.append(labels_batch.cpu())


# ============================================================
# Concatenate Predictions
# ============================================================

probabilities = torch.cat(all_probabilities, dim=0).numpy()
predicted = torch.cat(all_predictions, dim=0).numpy()
y_true = torch.cat(all_targets, dim=0).numpy()


# ============================================================
# Evaluation
# ============================================================

results = classification_report(
    y_true,
    predicted,
    digits=4,
    zero_division=0
)

print("\nClassification report:")
print(results)

accuracy = (predicted == y_true).mean()
balanced_accuracy = balanced_accuracy_score(y_true, predicted)
f1 = f1_score(y_true, predicted, zero_division=0)
auc = roc_auc_score(y_true, probabilities[:, 1])

print(f"Accuracy:          {accuracy:.4f}")
print(f"Balanced Accuracy: {balanced_accuracy:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"AUC:               {auc:.4f}")


# ============================================================
# Summary
# ============================================================

print("\n==============================================")
print("GPT2 TEXT + RANDOM OVERSAMPLING SUMMARY")
print("==============================================")

print("Model               : GPT2 Text")
print("Input               : Patient narrative")
print("Learning Rate       : 0.001")
print("Optimiser           : AdamW")
print("Class Weights       : None")
print("Random Oversampling : Yes")
print("SMOTE               : No")
print("GPT2 Frozen         : Yes")
print("Epochs              :", EPOCHS)
print("Best Epoch          :", best_epoch)
print("Best Test Loss      :", round(best_val_loss, 4))
print("Training Samples    :", len(train_dataset))
print("Test Patients       :", len(test_dataset))
print("Seed                :", SEED)

#Sol2 RandomOverSampler with weighted loss

In [ ]:
# ============================================================
# Imports
# ============================================================

import os
import random
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import gdown

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, balanced_accuracy_score, f1_score, roc_auc_score
from imblearn.over_sampling import RandomOverSampler
from transformers import GPT2Tokenizer, GPT2Model


# ============================================================
# Read Data
# ============================================================

gdown.download(
    id="1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q",
    output="clinical_data.json",
    quiet=False
)

gdown.download(
    id="1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F",
    output="pathological_data.json",
    quiet=False
)

with open("clinical_data.json", "r") as f:
    clinical = pd.DataFrame(json.load(f))

with open("pathological_data.json", "r") as f:
    pathology = pd.DataFrame(json.load(f))

df = clinical.merge(pathology, on="patient_id", how="inner")

print("Merged shape:", df.shape)
print("Clinical shape:", clinical.shape)
print("Pathology shape:", pathology.shape)


# ============================================================
# HPV Labels
# ============================================================

df = df[df["hpv_association_p16"].isin(["positive", "negative"])].copy()
df["HPV"] = df["hpv_association_p16"].map({"negative": 0, "positive": 1})

print("\nHPV distribution:")
print(df["HPV"].value_counts())


# ============================================================
# GPT2 Hancock Features
# ============================================================

features = [
    "age_at_initial_diagnosis",
    "sex",
    "smoking_status",
    "primarily_metastasis",
    "first_treatment_intent",
    "first_treatment_modality",
    "days_to_first_treatment",
    "adjuvant_treatment_intent",
    "adjuvant_radiotherapy",
    "adjuvant_radiotherapy_modality",
    "adjuvant_systemic_therapy",
    "adjuvant_systemic_therapy_modality",
    "adjuvant_radiochemotherapy",
    "primary_tumor_site",
    "pT_stage",
    "pN_stage",
    "histologic_type",
    "number_of_positive_lymph_nodes",
    "number_of_resected_lymph_nodes",
    "perinodal_invasion",
    "lymphovascular_invasion_L",
    "vascular_invasion_V",
    "perineural_invasion_Pn",
    "resection_status",
    "infiltration_depth_in_mm"
]


# ============================================================
# Missing Values
# ============================================================

for col in features:
    if df[col].dtype == object:
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())


# ============================================================
# Narrative Function
# ============================================================

def patient_to_text(row):
    text = (
        f"This patient was diagnosed at age {row['age_at_initial_diagnosis']}. "
        f"Sex is {row['sex']}. "
        f"Smoking status is {row['smoking_status']}. "
        f"Primary tumour site is {row['primary_tumor_site']}. "
        f"Histologic type is {row['histologic_type']}. "
        f"Pathological T stage is {row['pT_stage']}. "
        f"Pathological N stage is {row['pN_stage']}. "
        f"The patient has {row['number_of_positive_lymph_nodes']} positive lymph nodes and "
        f"{row['number_of_resected_lymph_nodes']} resected lymph nodes. "
        f"Perinodal invasion status is {row['perinodal_invasion']}. "
        f"Lymphovascular invasion status is {row['lymphovascular_invasion_L']}. "
        f"Vascular invasion status is {row['vascular_invasion_V']}. "
        f"Perineural invasion status is {row['perineural_invasion_Pn']}. "
        f"Resection status is {row['resection_status']}. "
        f"Infiltration depth is {row['infiltration_depth_in_mm']} mm. "
        f"First treatment intent was {row['first_treatment_intent']}. "
        f"First treatment modality was {row['first_treatment_modality']}. "
        f"Days to first treatment was {row['days_to_first_treatment']}. "
        f"Adjuvant treatment intent was {row['adjuvant_treatment_intent']}. "
        f"Adjuvant radiotherapy was {row['adjuvant_radiotherapy']}. "
        f"Adjuvant radiotherapy modality was {row['adjuvant_radiotherapy_modality']}. "
        f"Adjuvant systemic therapy was {row['adjuvant_systemic_therapy']}. "
        f"Adjuvant systemic therapy modality was {row['adjuvant_systemic_therapy_modality']}. "
        f"Adjuvant radiochemotherapy was {row['adjuvant_radiochemotherapy']}. "
        f"Predict whether the patient is HPV positive or HPV negative."
    )
    return text


# ============================================================
# Create GPT2 Text Corpus
# ============================================================

texts = df.apply(patient_to_text, axis=1).tolist()
labels = df["HPV"].tolist()

print("\nPatients:", len(texts))
print("\nOriginal class distribution:")
print(pd.Series(labels).value_counts())


# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


SEED = 42
seed_everything(SEED)


# ============================================================
# Device
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nUsing device:", device)


# ============================================================
# Train Test Split
# ============================================================

train_texts, test_texts, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.20,
    random_state=SEED,
    stratify=labels
)

print("\nTrain class distribution BEFORE oversampling:")
print(pd.Series(y_train).value_counts())

print("\nTest class distribution:")
print(pd.Series(y_test).value_counts())


# ============================================================
# Random Oversampling
# Training Data Only
# ============================================================

ros = RandomOverSampler(random_state=SEED)

train_indices = np.arange(len(train_texts)).reshape(-1, 1)
train_indices_resampled, y_train_resampled = ros.fit_resample(train_indices, y_train)

train_texts = [train_texts[index[0]] for index in train_indices_resampled]
y_train = y_train_resampled.tolist()

print("\nTrain class distribution AFTER oversampling:")
print(pd.Series(y_train).value_counts())

print("\nNumber of training samples after oversampling:", len(train_texts))


# ============================================================
# Tokenizer
# ============================================================

MODEL_NAME = "openai-community/gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

MAX_LENGTH = 256


# ============================================================
# Dataset
# ============================================================

class HPVTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoding = self.tokenizer(
            self.texts[index],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[index], dtype=torch.long)
        }


# ============================================================
# Datasets
# ============================================================

train_dataset = HPVTextDataset(
    texts=train_texts,
    labels=y_train,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

test_dataset = HPVTextDataset(
    texts=test_texts,
    labels=y_test,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

print("\nTraining dataset size:", len(train_dataset))
print("Test dataset size:", len(test_dataset))


# ============================================================
# DataLoaders
# ============================================================

BATCH_SIZE = 256

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


# ============================================================
# GPT2 Text Classifier
# ============================================================

class HPVNetGPT2Text(nn.Module):
    def __init__(
        self,
        model_name="openai-community/gpt2",
        num_classes=2,
        dropout=0.1,
        freeze_gpt2=True
    ):
        super().__init__()

        self.freeze_gpt2 = freeze_gpt2
        self.gpt2 = GPT2Model.from_pretrained(model_name)

        self.gpt2.config.use_cache = False
        self.gpt2.config.pad_token_id = tokenizer.pad_token_id

        hidden_size = self.gpt2.config.hidden_size

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

        if freeze_gpt2:
            self.gpt2.requires_grad_(False)

    def train(self, mode=True):
        super().train(mode)

        if self.freeze_gpt2:
            self.gpt2.eval()

        return self

    def forward(self, input_ids, attention_mask):
        outputs = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True
        )

        hidden_states = outputs.last_hidden_state

        # Masked mean pooling
        expanded_mask = attention_mask.unsqueeze(-1).expand_as(hidden_states).float()
        hidden_sum = (hidden_states * expanded_mask).sum(dim=1)
        valid_token_count = expanded_mask.sum(dim=1).clamp(min=1e-9)
        pooled_output = hidden_sum / valid_token_count

        logits = self.classifier(pooled_output)
        return logits


# ============================================================
# Model
# ============================================================

model = HPVNetGPT2Text(
    model_name=MODEL_NAME,
    num_classes=2,
    dropout=0.1,
    freeze_gpt2=True
).to(device)


# ============================================================
# Loss
# ============================================================

# criterion = nn.CrossEntropyLoss()

class_counts = np.bincount(y_train)

class_weights = (
    len(y_train)
    / (
        len(class_counts) *
        class_counts
    )
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


# ============================================================
# Optimiser
# ============================================================

optimizer = torch.optim.AdamW(
    filter(lambda parameter: parameter.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)

print(
    "\nTrainable parameters:",
    sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
)


# ============================================================
# Training
# ============================================================

EPOCHS = 200

best_val_loss = float("inf")
best_epoch = 0

for epoch in range(EPOCHS):

    # Training
    model.train()

    train_loss_sum = 0.0
    train_sample_count = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels_batch = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        train_loss = criterion(outputs, labels_batch)
        train_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            filter(lambda parameter: parameter.requires_grad, model.parameters()),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum += train_loss.item() * input_ids.size(0)
        train_sample_count += input_ids.size(0)

    average_train_loss = train_loss_sum / train_sample_count


    # Validation / Test
    model.eval()

    test_loss_sum = 0.0
    test_sample_count = 0

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels_batch = batch["label"].to(device, non_blocking=True)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            test_loss = criterion(outputs, labels_batch)

            test_loss_sum += test_loss.item() * input_ids.size(0)
            test_sample_count += input_ids.size(0)

    average_test_loss = test_loss_sum / test_sample_count


    # Save Best Model
    if average_test_loss < best_val_loss:
        best_val_loss = average_test_loss
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "best_model_gpt2_text_oversampling.pth"
        )


    # Progress
    if (epoch + 1) % 5 == 0:
        print(
            f"Epoch {epoch + 1:03d}, "
            f"Train={average_train_loss:.4f}, "
            f"Test={average_test_loss:.4f}, "
            f"Best Epoch={best_epoch}"
        )


print("\nBest Test Loss:", best_val_loss)
print("Best Epoch:", best_epoch)


# ============================================================
# Load Best Model
# ============================================================

checkpoint = torch.load(
    "best_model_gpt2_text_oversampling.pth",
    map_location=device
)

model.load_state_dict(checkpoint)
model.eval()


# ============================================================
# Inference
# ============================================================

all_probabilities = []
all_predictions = []
all_targets = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels_batch = batch["label"]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probabilities = torch.softmax(outputs, dim=1)
        predictions = torch.argmax(probabilities, dim=1)

        all_probabilities.append(probabilities.cpu())
        all_predictions.append(predictions.cpu())
        all_targets.append(labels_batch.cpu())


# ============================================================
# Concatenate Predictions
# ============================================================

probabilities = torch.cat(all_probabilities, dim=0).numpy()
predicted = torch.cat(all_predictions, dim=0).numpy()
y_true = torch.cat(all_targets, dim=0).numpy()


# ============================================================
# Evaluation
# ============================================================

results = classification_report(
    y_true,
    predicted,
    digits=4,
    zero_division=0
)

print("\nClassification report:")
print(results)

accuracy = (predicted == y_true).mean()
balanced_accuracy = balanced_accuracy_score(y_true, predicted)
f1 = f1_score(y_true, predicted, zero_division=0)
auc = roc_auc_score(y_true, probabilities[:, 1])

print(f"Accuracy:          {accuracy:.4f}")
print(f"Balanced Accuracy: {balanced_accuracy:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"AUC:               {auc:.4f}")


# ============================================================
# Summary
# ============================================================

print("\n==============================================")
print("GPT2 TEXT + RANDOM OVERSAMPLING SUMMARY")
print("==============================================")

print("Model               : GPT2 Text")
print("Input               : Patient narrative")
print("Learning Rate       : 0.001")
print("Optimiser           : AdamW")
print("Class Weights       : None")
print("Random Oversampling : Yes")
print("SMOTE               : No")
print("GPT2 Frozen         : Yes")
print("Epochs              :", EPOCHS)
print("Best Epoch          :", best_epoch)
print("Best Test Loss      :", round(best_val_loss, 4))
print("Training Samples    :", len(train_dataset))
print("Test Patients       :", len(test_dataset))
print("Seed                :", SEED)